# AMPS I · Week 13 — Lab
## Advection–diffusion: Galerkin oscillations, upwinding and SUPG

| | |
|---|---|
| **Lecture sections** | Part A (Péclet numbers, the 1D stencil, optimal diffusion, SUPG) · Part B (forms, test problems) · Part C (measurements) · Part D (LLM) |
| **Time** | ≈ 80 min |
| **You will** | find the Péclet threshold of Galerkin in 1D · implement SUPG and see it reproduce the exact nodal values · measure errors away from boundary layers in 2D · look at an interior layer · make $\tau$ safe where $\mathbf b=0$ |

$-\varepsilon\Delta u+\mathbf b\cdot\nabla u=f$, Dirichlet data on the whole boundary, P1 unless stated. Mesh Péclet number $\mathrm{Pe}_h=|\mathbf b|h/(2\varepsilon)$.

**Reminder.** `%%fenicsx` cells run in a separate MPI process; ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · 1D: when does Galerkin oscillate?  *(lecture §A2–A4)*
$-\varepsilon u''+u'=1$ on $(0,1)$, $u(0)=u(1)=0$, $N=20$ ($h=0.05$). Galerkin and full upwind for five values of $\varepsilon$.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def exact1d(X, eps):
    """-eps u'' + u' = 1, u(0) = u(1) = 0 - written with decaying exponentials only."""
    return X - (np.exp((X - 1) / eps) - np.exp(-1 / eps)) / (1 - np.exp(-1 / eps))

def adr1d(N, eps, method):
    """P1 on (0, 1), N cells. method: 'galerkin' or 'upwind' (artificial diffusion |b| h / 2). Returns sorted nodes and nodal values."""
    msh = mesh.create_unit_interval(MPI.COMM_SELF, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    h = ufl.CellDiameter(msh)
    a = (eps * u.dx(0) * v.dx(0) + u.dx(0) * v) * ufl.dx
    L = 1.0 * v * ufl.dx
    if method == "upwind":
        a += (h / 2) * u.dx(0) * v.dx(0) * ufl.dx
    dofs = fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0.0) | np.isclose(x[0], 1.0))
    uh = LinearProblem(a, L, bcs=[fem.dirichletbc(0.0, dofs, V)], petsc_options_prefix=f"a1d_{method}_{N}_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve()
    X = V.tabulate_dof_coordinates()[:, 0]
    o = np.argsort(X)
    return X[o], uh.x.array[o]

N = 20
for eps in (0.1, 0.02, 0.01, 0.005, 1e-4):
    Peh = (1.0 / N) / (2 * eps)
    row = []
    for m in ("galerkin", "upwind"):
        X, U = adr1d(N, eps, m)
        row.append(f"{m}: nodal error {np.abs(U - exact1d(X, eps)).max():.2e}, max u_h {U.max():.3f}")
    print(f"eps = {eps:6.0e}, Pe_h = {Peh:7.2f}, exact max {exact1d(np.linspace(0, 1, 100001), eps).max():.3f} | " + " | ".join(row))
prof = {m: adr1d(N, 0.005, m) for m in ("galerkin", "upwind")}
np.savez("/tmp/fnx/w13_1d.npz", Xg=prof["galerkin"][0], Ug=prof["galerkin"][1], Xu=prof["upwind"][0], Uu=prof["upwind"][1])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/w13_1d.npz")
Xf = np.linspace(0, 1, 2001); eps = 0.005
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(Xf, Xf - (np.exp((Xf - 1) / eps) - np.exp(-1 / eps)) / (1 - np.exp(-1 / eps)), "k-", label="exact")
ax.plot(d["Xg"], d["Ug"], "o--", color="C3", label="Galerkin"); ax.plot(d["Xu"], d["Uu"], "s-", color="C0", label="upwind")
ax.set_title("eps = 0.005, N = 20 (Pe_h = 5)"); ax.legend(); fig.tight_layout(); plt.show()

> **✏️ Your turn (10 min).** The 1D analysis (§A3) is for $f=0$: $-\varepsilon u''+u'=0$, $u(0)=0$, $u(1)=1$, whose exact solution increases monotonically from 0 to 1.
> Write a Galerkin solver for this problem ($N=20$) and test the prediction "oscillations iff $\mathrm{Pe}_h>1$" for $\varepsilon=0.03,\,0.026,\,0.024,\,0.02$: report $\min u_h$,
> whether $u_h$ is monotone, and compare $u_{19}$ with the prediction $u_{N-1}=\frac{r^{N-1}-1}{r^N-1}$, $r=\frac{1+\mathrm{Pe}_h}{1-\mathrm{Pe}_h}$.

In [ ]:
%%fenicsx -np 1
# TODO: Galerkin threshold Pe_h = 1 for -eps u'' + u' = 0, u(0) = 0, u(1) = 1

## 2 · SUPG in 1D  *(lecture §A4–A5)*
> **✏️ Your turn (10 min).** Add SUPG to `adr1d`: $\tau=\frac{h}{2|b|}\big(\coth\mathrm{Pe}_h-1/\mathrm{Pe}_h\big)$, $\mathrm{Pe}_h=|b|h/(2\varepsilon)$, residual $-\varepsilon u''+u'-f$ times
> $\tau\,v'$ (for P1, $u''=0$ in each cell). Report the maximal nodal error for the five $\varepsilon$ of §1. What do you observe, and why (§A4)?

In [ ]:
%%fenicsx -np 1
# TODO: SUPG in 1D

## 3 · 2D boundary layers: where is the error?  *(lecture §B2, §C2)*
$\mathbf b=(2,3)$, exact solution with layers at $x=1$ and $y=1$ (lecture §B2), $\varepsilon=10^{-4}$, $N=16,\dots,128$. "away" = $L^2$ error on $[0,0.8]^2$.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def adr(msh, b, eps, f, method, deg=1, g=0.0, tau_scale=1.0, prefix="adr_"):
    """-eps Lap u + b.grad u = f in 2D, u = g on the whole boundary (g: number or numpy callable).
    method: 'galerkin', 'upwind' (isotropic artificial diffusion |b| h / 2) or 'supg'. Returns u_h and the KSP reason."""
    V = fem.functionspace(msh, ("Lagrange", deg))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    e = fem.Constant(msh, eps)
    h = ufl.CellDiameter(msh) / deg
    bn = ufl.sqrt(ufl.dot(b, b))
    a = (e * ufl.inner(ufl.grad(u), ufl.grad(v)) + ufl.dot(b, ufl.grad(u)) * v) * ufl.dx
    L = f * v * ufl.dx
    if method == "supg":
        Peh = bn * h / (2 * e)
        tau = tau_scale * h / (2 * bn) * (1 / ufl.tanh(Peh) - 1 / Peh)
        R = -e * ufl.div(ufl.grad(u)) + ufl.dot(b, ufl.grad(u))
        a += tau * R * ufl.dot(b, ufl.grad(v)) * ufl.dx
        L += tau * f * ufl.dot(b, ufl.grad(v)) * ufl.dx
    if method == "upwind":
        a += (bn * h / 2) * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
    msh.topology.create_connectivity(1, 2)
    dofs = fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology))
    if callable(g):
        gb = fem.Function(V); gb.interpolate(g); bc = fem.dirichletbc(gb, dofs)
    else:
        bc = fem.dirichletbc(float(g), dofs, V)
    pr = LinearProblem(a, L, bcs=[bc], petsc_options_prefix=prefix,
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    uh = pr.solve()
    return uh, pr.solver.getConvergedReason()

def layer_problem(N, eps, method, deg=1):
    """b = (2, 3), exact solution with layers at x = 1 and y = 1. Returns L2 error, L2 error on [0, 0.8]^2, min u_h, max u_h."""
    msh = mesh.create_unit_square(comm, N, N)
    x = ufl.SpatialCoordinate(msh)
    b = ufl.as_vector([2.0, 3.0])
    X, Y = x[0], x[1]
    uex = X * Y**2 - Y**2 * ufl.exp(2 * (X - 1) / eps) - X * ufl.exp(3 * (Y - 1) / eps) + ufl.exp((2 * (X - 1) + 3 * (Y - 1)) / eps)
    f = -eps * ufl.div(ufl.grad(uex)) + ufl.dot(b, ufl.grad(uex))
    uh, r = adr(msh, b, eps, f, method, deg, prefix=f"lay_{method}_{N}_{deg}_")
    dxq = ufl.dx(metadata={"quadrature_degree": 8})
    chi = ufl.conditional(ufl.And(ufl.lt(X, 0.8), ufl.lt(Y, 0.8)), 1.0, 0.0)
    S = lambda e_: np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(e_)), op=MPI.SUM))
    return (S((uh - uex)**2 * dxq), S(chi * (uh - uex)**2 * dxq),
            comm.allreduce(uh.x.array.min(), op=MPI.MIN), comm.allreduce(uh.x.array.max(), op=MPI.MAX))

eps = 1e-4
for m in ("galerkin", "upwind", "supg"):
    E = np.array([layer_problem(N, eps, m) for N in (16, 32, 64, 128)])
    say(f"{m:8s} L2 {[f'{e:.2e}' for e in E[:, 0]]} rates {rate(E[:, 0])}")
    say(f"{'':8s} away {[f'{e:.2e}' for e in E[:, 1]]} rates {rate(E[:, 1])} | u_h in [{E[-1, 2]:.3f}, {E[-1, 3]:.3f}] at N = 128")

> **✏️ Your turn (5 min).** SUPG with **P2** (`deg=2`; the helper then uses $h/2$ in $\tau$ and keeps $-\varepsilon\Delta u_h$ in the residual): away-error rates? Predict first (§A6).

In [ ]:
%%fenicsx -np 2
# TODO: P2 SUPG rates

## 4 · An interior layer: skew advection  *(lecture §B3, §C5)*
$\varepsilon=10^{-8}$; the exact solution lies in $[0,1]$. $N=32$, three methods (serial); fields saved for plotting.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def adr(msh, b, eps, f, method, deg=1, g=0.0, tau_scale=1.0, prefix="adr_"):
    """-eps Lap u + b.grad u = f in 2D, u = g on the whole boundary (g: number or numpy callable).
    method: 'galerkin', 'upwind' (isotropic artificial diffusion |b| h / 2) or 'supg'. Returns u_h and the KSP reason."""
    V = fem.functionspace(msh, ("Lagrange", deg))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    e = fem.Constant(msh, eps)
    h = ufl.CellDiameter(msh) / deg
    bn = ufl.sqrt(ufl.dot(b, b))
    a = (e * ufl.inner(ufl.grad(u), ufl.grad(v)) + ufl.dot(b, ufl.grad(u)) * v) * ufl.dx
    L = f * v * ufl.dx
    if method == "supg":
        Peh = bn * h / (2 * e)
        tau = tau_scale * h / (2 * bn) * (1 / ufl.tanh(Peh) - 1 / Peh)
        R = -e * ufl.div(ufl.grad(u)) + ufl.dot(b, ufl.grad(u))
        a += tau * R * ufl.dot(b, ufl.grad(v)) * ufl.dx
        L += tau * f * ufl.dot(b, ufl.grad(v)) * ufl.dx
    if method == "upwind":
        a += (bn * h / 2) * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
    msh.topology.create_connectivity(1, 2)
    dofs = fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology))
    if callable(g):
        gb = fem.Function(V); gb.interpolate(g); bc = fem.dirichletbc(gb, dofs)
    else:
        bc = fem.dirichletbc(float(g), dofs, V)
    pr = LinearProblem(a, L, bcs=[bc], petsc_options_prefix=prefix,
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    uh = pr.solve()
    return uh, pr.solver.getConvergedReason()

def skew(N, method):
    """eps = 1e-8, b = (cos(-pi/3), sin(-pi/3)), f = 0, u = 1 on y = 1 and on {x = 0, y > 0.7}, 0 elsewhere (serial)."""
    msh = mesh.create_unit_square(MPI.COMM_SELF, N, N)
    th = -np.pi / 3
    return adr(msh, ufl.as_vector([np.cos(th), np.sin(th)]), 1e-8, fem.Constant(msh, 0.0), method,
               g=lambda x: np.where(np.isclose(x[1], 1.0) | (np.isclose(x[0], 0.0) & (x[1] > 0.7)), 1.0, 0.0), prefix=f"skew_{method}_{N}_")

out = {}
for m in ("galerkin", "upwind", "supg"):
    uh, r = skew(32, m)
    V = uh.function_space
    out[m] = uh.x.array.copy(); out["X"] = V.tabulate_dof_coordinates()[:, :2]; out["tri"] = V.dofmap.list.copy()
    print(f"{m:8s}: reason {r}, u_h in [{uh.x.array.min():.3f}, {uh.x.array.max():.3f}]")
np.savez("/tmp/fnx/w13_skew.npz", **out)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

d = np.load("/tmp/fnx/w13_skew.npz")
t = mtri.Triangulation(d["X"][:, 0], d["X"][:, 1], d["tri"])
fig, axs = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, m in zip(axs, ("galerkin", "upwind", "supg")):
    tp = ax.tripcolor(t, np.clip(d[m], -0.2, 1.2), shading="gouraud", cmap="RdBu_r", vmin=-0.2, vmax=1.2)
    ax.set_aspect("equal"); ax.set_title(f"{m}: [{d[m].min():.2f}, {d[m].max():.3g}]")
fig.colorbar(tp, ax=axs, fraction=0.02); plt.show()

> **✏️ Your turn (5 min).** For SUPG, find **where** the minimum and the maximum of $u_h$ occur ($N=16,32,64$) and whether the range shrinks with $h$.
> Which layer produces the overshoot, which the undershoot?

In [ ]:
%%fenicsx -np 1
# TODO: location of the SUPG extremes

## 5 · A $\tau$ that survives $\mathbf b=0$  *(lecture §D1)*
$\mathbf b=(1,0)$ for $x>0.5$ and $\mathbf b=0$ for $x<0.5$, $\varepsilon=10^{-3}$, $f=1$, $N=16$: the textbook $\tau$ divides by $|\mathbf b|$ and by $\mathrm{Pe}_h$.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def adr(msh, b, eps, f, method, deg=1, g=0.0, tau_scale=1.0, prefix="adr_"):
    """-eps Lap u + b.grad u = f in 2D, u = g on the whole boundary (g: number or numpy callable).
    method: 'galerkin', 'upwind' (isotropic artificial diffusion |b| h / 2) or 'supg'. Returns u_h and the KSP reason."""
    V = fem.functionspace(msh, ("Lagrange", deg))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    e = fem.Constant(msh, eps)
    h = ufl.CellDiameter(msh) / deg
    bn = ufl.sqrt(ufl.dot(b, b))
    a = (e * ufl.inner(ufl.grad(u), ufl.grad(v)) + ufl.dot(b, ufl.grad(u)) * v) * ufl.dx
    L = f * v * ufl.dx
    if method == "supg":
        Peh = bn * h / (2 * e)
        tau = tau_scale * h / (2 * bn) * (1 / ufl.tanh(Peh) - 1 / Peh)
        R = -e * ufl.div(ufl.grad(u)) + ufl.dot(b, ufl.grad(u))
        a += tau * R * ufl.dot(b, ufl.grad(v)) * ufl.dx
        L += tau * f * ufl.dot(b, ufl.grad(v)) * ufl.dx
    if method == "upwind":
        a += (bn * h / 2) * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
    msh.topology.create_connectivity(1, 2)
    dofs = fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology))
    if callable(g):
        gb = fem.Function(V); gb.interpolate(g); bc = fem.dirichletbc(gb, dofs)
    else:
        bc = fem.dirichletbc(float(g), dofs, V)
    pr = LinearProblem(a, L, bcs=[bc], petsc_options_prefix=prefix,
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    uh = pr.solve()
    return uh, pr.solver.getConvergedReason()

msh = mesh.create_unit_square(MPI.COMM_SELF, 16, 16)
x = ufl.SpatialCoordinate(msh)
b = ufl.as_vector([ufl.conditional(ufl.lt(x[0], 0.5), 0.0, 1.0), 0.0])
uh, r = adr(msh, b, 1e-3, fem.Constant(msh, 1.0), "supg", prefix="bzero_")
print(f"textbook tau: KSP reason {r}, all values finite: {bool(np.all(np.isfinite(uh.x.array)))}")

> **✏️ Your turn (10 min).** Rewrite $\tau$ so that it never divides by $|\mathbf b|$ or by zero, and reduces to $h^2/(12\varepsilon)$ as $\mathrm{Pe}_h\to0$ (use $\xi(z)\approx z/3$).
> Solve again: reason, finite values, $\max u_h$. Then check on the 2D layer problem of §3 ($\varepsilon=10^{-4}$, $N=32$) that your $\tau$ gives the same error as the textbook one.

In [ ]:
%%fenicsx -np 1
# TODO: a safe tau

## 6 · LLM-assisted coding: live demo  *(lecture §D1)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11, one %%fenicsx -np 2 cell. Solve -eps*Laplace(u) + b.grad(u) = f on the unit square with P1 elements and SUPG
stabilisation, b = (2, 3), eps = 1e-4, u = 0 on the whole boundary, with the manufactured solution
u = x*y^2 - y^2*exp(2(x-1)/eps) - x*exp(3(y-1)/eps) + exp((2(x-1)+3(y-1))/eps) (compute f with UFL).
For N = 16, 32, 64 print the L2 error on [0, 0.8]^2 and the range of u_h, on rank 0.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] The same code with $\varepsilon$ large ($\mathrm{Pe}_h\ll1$) agrees with plain Galerkin
- [ ] In 1D the nodal values are exact (§2)
- [ ] Away-error rate 2 for P1, $\max u_h$ a little above 1 at the layer — not 2, not 0.5
- [ ] The exact solution contains no $e^{+1/\varepsilon}$ (overflow)
- [ ] The result does not change between `-np 1` and `-np 2`